In [ ]:
import gradio as gr
from openai import OpenAI

from week2.scraper import fetch_website_contents

openai = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

In [ ]:

def greet(name):
  return "Hello " + name + "!"

demo = gr.Interface(fn=greet, inputs="textbox", outputs="textbox", flagging_mode="never")

# demo.launch()

In [ ]:
system_message = "You are a helpful assistant"

def message_llama(prompt):
  messages = [{"role": "system", "content": system_message}, {"role": "user", "content": prompt}]
  response = openai.chat.completions.create(model="llama3.1:8b", messages=messages)
  return response.choices[0].message.content

# message_llama("What is today's date?")

In [ ]:
message_input = gr.Textbox(label="Your message:", info="Enter a message for Llama 3.2", lines=7)
message_output = gr.Textbox(label="Response:", lines=8)

view = gr.Interface(
  fn=message_llama,
  title="Llama",
  inputs=[message_input],
  outputs=[message_output],
  examples=["hello", "howdy"],
  flagging_mode="never"
)

# view.launch()

In [ ]:
def stream_gemma(prompt):
  messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": prompt}
  ]
  stream = openai.chat.completions.create(
    model="gemma4:e4b",
    messages=messages,
    stream=True
  )
  result = ""
  for chunk in stream:
    result += chunk.choices[0].delta.content or ""
    yield result

In [ ]:
system_message = "You are a helpful assistant that responds in markdown without code blocks"

message_input = gr.Textbox(label="Your message:", info="Enter a message for Llama 3.1", lines=7)
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
  fn=stream_gemma,
  title="Gemma",
  inputs=[message_input],
  outputs=[message_output],
  examples=[
    "Explain the Transformer architecture to a layperson",
    "Explain the Transformer architecture to an aspiring AI engineer",
  ],
  flagging_mode="never"
)

# view.launch()

In [ ]:
def stream_llama(prompt):
  messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": prompt}
  ]
  stream = openai.chat.completions.create(
    model="llama3.1:8b",
    messages=messages,
    stream=True
  )
  result = ""
  for chunk in stream:
    result += chunk.choices[0].delta.content or ""
    yield result

In [ ]:
system_message = """
  You are an assistant that analyzes the contents of a company website landing page
  and creates a short brochure about the company for prospective customers, investors and recruits.
  Respond in markdown without code blocks.
"""

def stream_brochure(company_name, url, model):
  yield ""
  prompt = f"Please generate a company brochure for {company_name}. Here is their landing page:\n"
  prompt += fetch_website_contents(url)
  if model=="GEMMA":
      result = stream_gemma(prompt)
  elif model=="LLAMA":
      result = stream_llama(prompt)
  else:
      raise ValueError("Unknown model")
  yield from result

In [ ]:
name_input = gr.Textbox(label="Company name:")
url_input = gr.Textbox(label="Landing page URL including http:// or https://")
model_selector = gr.Dropdown(["GEMMA", "LLAMA"], label="Select model", value="GEMMA")
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
  fn=stream_brochure,
  title="Brochure Generator",
  inputs=[name_input, url_input, model_selector],
  outputs=[message_output],
  examples=[
      ["Hugging Face", "https://huggingface.co", "GEMMA"],
      ["Edward Donner", "https://edwarddonner.com", "LLAMA"]
  ],
  flagging_mode="never"
)

view.launch()